# Vol-method comparison: SMA(252) vs blended(32/2520)

Re-runs the full strategy scan with two `FixedRiskSizer` volatility models and compares the diversified equal-weight portfolio per strategy.

Findings: the blended model (70% short span 32, 30% long span 2520) raises Sharpe for nearly every positive strategy and lowers portfolio volatility — the very slow long span anchors the vol estimate, so positions change less and survive vol spikes.

In [ ]:
"""Compare FixedRiskSizer vol methods: sma(252) vs blended(32/2520)."""
import pandas as pd
from pathlib import Path

from sysstrat import (
    Asset, Capital, FixedRiskSizer, BacktestRunner, PortfolioRunner,
    BuyAndHoldStrategy, MACrossoverStrategy, EWMACStrategy, NormalisedTrendStrategy,
    BreakoutStrategy, AccelerationStrategy, SkewStrategy, MeanReversionStrategy,
    TimeSeriesMomentumStrategy, DonchianStrategy, BollingerMeanReversionStrategy,
    MACDStrategy, RSIMeanReversionStrategy,
    load_simple_price_csv,
)

ROOT = Path.cwd()
while not (ROOT / "data" / "MCFTR.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"

INSTRUMENTS = {
    "MCFTR": "MCFTR.csv", "RGBITR": "RGBITR.csv", "GLDRUB": "GLDRUB_TOM.csv",
    "CNYRUB": "CNYRUB_TOM.csv", "USDRUB": "USDRUB.csv",
}
assets = {
    t: Asset(ticker=t, price_data=load_simple_price_csv(DATA_DIR / f),
             commission_rate=0.0004, slippage_rate=0.001)
    for t, f in INSTRUMENTS.items()
}
start = max(a.price_data.index.min() for a in assets.values())
end = min(a.price_data.index.max() for a in assets.values())
assets = {t: a.slice(start, end) for t, a in assets.items()}

capital = Capital(initial_capital=100_000)

SIZERS = {
    "sma(252)": FixedRiskSizer(risk_target=0.20, vol_method="sma", volatility_window=252, max_leverage=1.0),
    "blended(32/2520)": FixedRiskSizer(risk_target=0.20, vol_method="blended", short_span=32, long_span=2520, max_leverage=1.0),
}

STRATEGIES = {
    "Buy & Hold": BuyAndHoldStrategy(),
    "MA Cross (10/50)": MACrossoverStrategy(short_window=10, long_window=50),
    "MA Cross LS": MACrossoverStrategy(short_window=10, long_window=50, mode="long_short"),
    "EWMAC (16/64)": EWMACStrategy(),
    "EWMAC (32/128)": EWMACStrategy(fast_window=32, slow_window=128),
    "Normalised Trend": NormalisedTrendStrategy(),
    "Breakout (40)": BreakoutStrategy(horizon=40),
    "Breakout (160)": BreakoutStrategy(horizon=160),
    "Acceleration": AccelerationStrategy(),
    "Skew": SkewStrategy(),
    "Mean Reversion": MeanReversionStrategy(),
    "TSMOM (252/21)": TimeSeriesMomentumStrategy(),
    "Donchian (55/20)": DonchianStrategy(),
    "Bollinger MR (40,2)": BollingerMeanReversionStrategy(),
    "MACD (12/26/9)": MACDStrategy(),
    "RSI(2) MR": RSIMeanReversionStrategy(),
}

rows = []
for sname, strategy in STRATEGIES.items():
    row = {"strategy": sname}
    for szname, sizer in SIZERS.items():
        reports = {t: BacktestRunner(capital, a, sizer).run(strategy) for t, a in assets.items()}
        m = PortfolioRunner(capital).run(reports).metrics
        row[f"{szname} sharpe"] = round(m.sharpe_ratio, 2)
        row[f"{szname} ret%"] = round(m.total_return_pct, 1)
        row[f"{szname} vol%"] = round(m.annual_volatility_pct, 2)
        row[f"{szname} maxDD%"] = round(m.max_drawdown_pct, 1)
    rows.append(row)

df = pd.DataFrame(rows).set_index("strategy")
df["sharpe delta"] = df["blended(32/2520) sharpe"] - df["sma(252) sharpe"]
df = df.sort_values("blended(32/2520) sharpe", ascending=False)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
print("=== Diversified equal-weight portfolio: sizer comparison ===")
print(df.to_string())

# Per-asset Buy & Hold under both sizers
print()
print("=== Per-asset Buy & Hold Sharpe ===")
arows = []
for t, a in assets.items():
    row = {"asset": t}
    for szname, sizer in SIZERS.items():
        m = BacktestRunner(capital, a, sizer).run(BuyAndHoldStrategy()).metrics
        row[szname] = round(m.sharpe_ratio, 2)
    arows.append(row)
print(pd.DataFrame(arows).set_index("asset").to_string())
